In [1]:
from factor_analysis_v0 import *
from pathlib import Path

data_config = DataConfig(
    underlying_gex_path=r"E:\Pythonfiles\ProjectGamma\results\option_gamma\underlying_gex_daily_with_permno.parquet",
    crsp_daily_path=r"E:\Pythonfiles\ProjectGamma\data\crsp_daily_common_2018_2024_linked_permnos.parquet",
    start_date="2018-01-01",
    end_date="2024-12-31",
    min_abs_price=1.0,
)

column_config = ColumnConfig(
    id_col="permno",
    date_col="date",
    gex_col="net_gex_1pct",
    ret_col="ret",
    spot_col="spot",
    crsp_price_col="prc",
    volume_col="vol",
    control_cols=[
        "price_stock",
        "market_equity_crsp",
        "vol",
        "shrout",
        "spot",
        "total_open_interest",
        "total_option_volume",
    ],
)

preprocess_config = PreprocessConfig(
    winsorize=True,
    winsor_lower=0.01,
    winsor_upper=0.99,
    cross_sectional_zscore=True,
    zscore_suffix="_z",
    min_cross_section_size=10,
    missing_threshold=0.95,
    allow_missing_factors=True,
    use_abs_crsp_price=True,
)

target_config = TargetConfig(
    horizons=[1, 5, 20],
    create_signed_return=True,
    create_abs_return=True,
    create_squared_return=True,
    create_realized_vol=True,
    create_downside_semivar=True,
    create_tail_indicators=True,
    tail_quantiles=[0.05, 0.95],
    tail_groupby="by_date",
    return_type="simple",
)

output_config = OutputConfig(
    output_dir=r"E:\Pythonfiles\ProjectGamma\results\gex_collab_phase1",
    save_panel_snapshot=True,
    save_metadata=True,
    panel_snapshot_name="phase1_panel.parquet",
    metadata_name="phase1_metadata.json",
    verbose=True,
)

"""factor_specs = [
    FactorSpec(
        name="bs_factor_csv",
        path=r"data/factors/BS.csv",
        file_type="csv",
        id_col="PERMNO",
        date_col="date",
        source_id_type="permno",
        frequency="daily",
        factor_cols=[
            "n", "ret", "alpha", "b_mkt", "b_smb", "b_hml", "b_umd",
            "ivol", "tvol", "R2", "exret"
        ],
        numeric_only=False,
        suffix="__bs",
        lag_days=0,
    ),
]"""

identifier_config = IdentifierConfig(
    mapping_path=r"data/secid_crsp_name_mapping_2018_2024.csv",
    mapping_file_type="csv",
    permno_col="permno",
    secid_col="secid",
    ticker_col="ticker",
    cusip_col="cusip",
    ncusip_col="ncusip",
    link_method_col="link_method",
    duplicate_resolution="drop_ambiguous",
)

factor_specs = [
    FactorSpec(
        name="mii_factor_csv",
        path=r"data/factors/MII.csv",
        file_type="csv",
        id_col="SYM_ROOT",
        date_col="DATE",
        source_id_type="ticker",
        frequency="daily",
        factor_cols=[
            # quoted spread / depth
            "QuotedSpread_Dollar_tw",
            "QuotedSpread_Percent_tw",
            "BestOfrDepth_Dollar_tw",
            "BestBidDepth_Dollar_tw",
            "BestOfrDepth_Share_tw",
            "BestBidDepth_Share_tw",

            # effective spread
            "EffectiveSpread_Dollar_Ave",
            "EffectiveSpread_Percent_Ave",
            "EffectiveSpread_Dollar_DW",
            "EffectiveSpread_Dollar_SW",
            "EffectiveSpread_Percent_DW",
            "EffectiveSpread_Percent_SW",

            # realized spread
            "DollarRealizedSpread_LR_Ave",
            "PercentRealizedSpread_LR_Ave",
            "DollarRealizedSpread_LR_SW",
            "DollarRealizedSpread_LR_DW",
            "PercentRealizedSpread_LR_SW",
            "PercentRealizedSpread_LR_DW",

            # price impact
            "DollarPriceImpact_LR_Ave",
            "PercentPriceImpact_LR_Ave",
            "DollarPriceImpact_LR_SW",
            "DollarPriceImpact_LR_DW",
            "PercentPriceImpact_LR_SW",
            "PercentPriceImpact_LR_DW",

            # intraday volatility
            "ivol_t",
            "ivol_q",

            # order imbalance / signed flow
            "bs_ratio_num",
            "bs_ratio_vol",
            "TSignSqrtDVol1",
            "TSignSqrtDVol2",

            # price informativeness / concentration / variance ratio
            "HIndex",
            "var_ratio1",
            "var_ratio2",
            "var_ratio3",
            "var_ratio4",
            "var_ratio5",

            # retail flow imbalance
            "bs_ratio_retail_num",
            "bs_ratio_retail_vol",

            # institutional flow imbalance
            "bs_ratio_Inst20k_num",
            "bs_ratio_Inst20k_vol",
            "bs_ratio_Inst50k_num",
            "bs_ratio_Inst50k_vol",
        ],
        numeric_only=False,
        suffix="__mii",
        lag_days=0,
    ),
]

regression_config = RegressionConfig(
    use_fama_macbeth=True,
    nw_lags=5,
    add_intercept=True,
    use_controls=True,
    control_cols=[
        "price_stock",
        "market_equity_crsp",
        "vol",
        "shrout",
        "spot",
        "total_open_interest",
        "total_option_volume",
    ],
    min_obs_per_date=10,
    min_dates_required=20,
    regime_validation_y_cols=[
        "ret_fwd_1d",
        "abs_ret_fwd_1d",
        "rv_fwd_5d",
        "tail_left_fwd_1d",
    ],
    interaction_y_cols=[
        "abs_ret_fwd_1d",
        "rv_fwd_5d",
        "tail_left_fwd_1d",
    ],
    n_buckets=3,
)

classification_config = ClassificationConfig(
    enabled=True,
    target_cols=[
        "tail_left_fwd_1d",
        "tail_left_fwd_5d",
        "tail_abs_fwd_1d",
    ],
    train_end="2022-12-31",
    test_start="2023-01-01",
    test_end="2024-12-31",
    test_size=0.3,
    min_train_rows=200,
    min_test_rows=100,
    model_names=[
        "logit_l2",
        "logit_elasticnet",
    ],
    max_iter=5000,
    C=0.5,
    l1_ratio=0.3,
    class_weight="balanced",
    use_controls=True,
    control_cols=[
        "price_stock",
        "market_equity_crsp",
        "vol",
        "total_open_interest",
        "total_option_volume",
    ],
    include_factor_interaction=True,
    imputer_strategy="median",
)

exp = GEXCollaborativeEffectExperiment(
    data_config=data_config,
    column_config=column_config,
    preprocess_config=preprocess_config,
    target_config=target_config,
    regression_config=regression_config,
    classification_config=classification_config,
    output_config=output_config,
    identifier_config=identifier_config,
)

In [2]:
artifacts = exp.run_phase1(
    factor_specs=factor_specs,
    selected_factors=None,
)

panel = artifacts["panel"]
print(panel.head())
print(panel.columns.tolist()[:100])
print(artifacts["metadata"])

E:\Pythonfiles\ProjectGamma\factor_analysis_v0.py:1115: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)
E:\Pythonfiles\ProjectGamma\factor_analysis_v0.py:838: UserWarning: FactorSpec 'mii_factor_csv': dropped 1117 rows because ticker could not be mapped to permno.
  warnings.warn(


    secid  permno       date   spot  spot_close  spot_return  spot_cfadj  \
0  107525   10107 2018-01-02  85.95       85.95     0.004793        16.0   
1  107525   10107 2018-01-03  86.35       86.35     0.004654        16.0   
2  107525   10107 2018-01-04  87.11       87.11     0.008801        16.0   
3  107525   10107 2018-01-05  88.19       88.19     0.012398        16.0   
4  107525   10107 2018-01-08  88.28       88.28     0.001020        16.0   

   spot_shrout  n_contracts  n_optionids  ...  rv_fwd_20d  \
0    7714590.0          726          726  ...    0.011532   
1    7714590.0          754          754  ...    0.011620   
2    7714590.0          758          758  ...    0.012875   
3    7714590.0          735          735  ...    0.015585   
4    7714590.0          752          752  ...    0.017732   

   downside_semivar_fwd_20d  tail_left_fwd_20d  tail_right_fwd_20d  \
0                  0.000019                  0                   0   
1                  0.000022         

In [3]:
artifacts_phase2 = exp.run_phase2(
    factor_cols=None,
)

E:\Pythonfiles\RiskAnalysis7002\venv\Lib\site-packages\statsmodels\regression\linear_model.py:1782: RuntimeWarning: invalid value encountered in scalar divide
  return 1 - self.ssr/self.centered_tss
E:\Pythonfiles\RiskAnalysis7002\venv\Lib\site-packages\statsmodels\regression\linear_model.py:1782: RuntimeWarning: invalid value encountered in scalar divide
  return 1 - self.ssr/self.centered_tss
E:\Pythonfiles\RiskAnalysis7002\venv\Lib\site-packages\statsmodels\regression\linear_model.py:1782: RuntimeWarning: invalid value encountered in scalar divide
  return 1 - self.ssr/self.centered_tss
E:\Pythonfiles\RiskAnalysis7002\venv\Lib\site-packages\statsmodels\regression\linear_model.py:1782: RuntimeWarning: invalid value encountered in scalar divide
  return 1 - self.ssr/self.centered_tss
E:\Pythonfiles\RiskAnalysis7002\venv\Lib\site-packages\statsmodels\regression\linear_model.py:1782: RuntimeWarning: invalid value encountered in scalar divide
  return 1 - self.ssr/self.centered_tss
E:\Py

In [ ]:
artifacts_phase3 = exp.run_phase3(
    factor_cols=None,
    regime_split_y_col="ret_fwd_1d",
    double_sort_y_cols=[
        "ret_fwd_1d",
        "abs_ret_fwd_1d",
        "rv_fwd_5d",
        "tail_left_fwd_1d",
    ],
)

In [5]:
artifacts_phase4 = exp.run_phase4(
    factor_cols=None,
    target_cols=[
        "tail_left_fwd_1d",
        "tail_left_fwd_5d",
        "tail_abs_fwd_1d",
    ],
)